In [0]:
# ============================================================
# PARAMETRI - Adatta questi valori al tuo ambiente
# ============================================================
config_path_yaml = "../../src/config/config_setup.yml"
# percorso_yml_generato = "./Config_Setup_Yml_Generato.yml"
percorso_yml_generato = "./Config_Environment_Yml_Generato.yml"

In [0]:
import os
import yaml
import re
import json

def crea_file_vuoto_workspace(percorso: str) -> bool:
    """
    Crea un file di testo vuoto (o lo tronca se esiste già).
    Crea anche le directory intermedie se mancanti.
    """
    try:
        directory = os.path.dirname(percorso)
        if directory and not os.path.exists(directory):
            os.makedirs(directory)
            print(f"Directory creata: {directory}")
        with open(percorso, 'w') as f:
            pass
        print(f"File vuoto creato/azzerato: {percorso}")
        return True
    except Exception as e:
        print(f"Errore durante la creazione del file: {e}")
        return False


In [0]:
def read_yaml_file(file_path: str) -> dict:
    """
    Legge un file YAML e restituisce il contenuto come dizionario.
    Supporta path DBFS (dbfs:/) e path Workspace (/Workspace/...).
    """
    if file_path.startswith("dbfs:/"):
        file_path = file_path.replace("dbfs:/", "/dbfs/")
    with open(file_path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)


def get_yaml_value(yaml_content: dict, key: str, default=None):
    """
    Estrae il valore di una chiave dal contenuto YAML.
    Supporta notazione con punto per chiavi annidate (es: 'database.host').
    """
    if not isinstance(yaml_content, dict):
        raise TypeError("yaml_content deve essere un dizionario")
    current_value = yaml_content
    try:
        for k in key.split('.'):
            if isinstance(current_value, dict):
                current_value = current_value[k]
            else:
                return default
        return current_value
    except (KeyError, TypeError):
        return default


def get_yaml_keys(yaml_content: dict, include_nested: bool = False, separator: str = '.') -> list:
    """
    Restituisce l'elenco delle chiavi del contenuto YAML.
    Con include_nested=True include anche le chiavi annidate (notazione punto).
    """
    if not isinstance(yaml_content, dict):
        raise TypeError("yaml_content deve essere un dizionario")
    if not include_nested:
        return list(yaml_content.keys())

    def _nested(data, parent=''):
        keys = []
        for k, v in data.items():
            full = f"{parent}{separator}{k}" if parent else k
            keys.append(full)
            if isinstance(v, dict):
                keys.extend(_nested(v, full))
        return keys

    return _nested(yaml_content)


In [0]:
def normalize_type(spark_type: str) -> str:
    """Converte un tipo Spark (es. StringType()) nel tipo SQL equivalente (es. STRING)."""
    spark_type = spark_type.strip()

    def convert_type(s):
        s = s.strip()
        s_lower = s.lower()
        if s_lower.startswith("decimaltype"):
            match = re.match(r"decimaltype\((\d+),\s*(\d+)\)", s_lower)
            if match:
                return f"DECIMAL({match.group(1)},{match.group(2)})"
        if s_lower.startswith("maptype"):
            match = re.match(r"maptype\((.*),\s*(.*),\s*(true|false)\)", s_lower, re.IGNORECASE)
            if match:
                return f"MAP<{convert_type(match.group(1))},{convert_type(match.group(2))}>"
        if s_lower.startswith("arraytype"):
            match = re.match(r"arraytype\((.*)\)", s_lower)
            if match:
                return f"ARRAY<{convert_type(match.group(1))}>"
        simple_map = {
            "stringtype()": "STRING",
            "integertype()": "INT",
            "longtype()": "BIGINT",
            "doubletype()": "DOUBLE",
            "timestamptype()": "TIMESTAMP",
            "datetype()": "DATE",
            "booleantype()": "BOOLEAN",
            "binarytype()": "BINARY",
            "floattype()": "FLOAT",
        }
        return simple_map.get(s_lower, s.upper())

    return convert_type(spark_type)


def validate_yaml_mapping(config: dict) -> list:
    """Valida la struttura YAML di una tabella rispetto alle convezioni attese."""
    errori = []
    allowed_prefixes = {"train", "infer", "out", "log", "util", "ref", "met"}
    campi_tecnici = {
        "iid_RecordID", "des_BusinessKey", "flg_lastSnap", "dat_startDate",
        "dat_endDate", "systemDate", "runId"
    }
    codice_prodotto = config.get("environment", {}).get("schema", "")
    if not codice_prodotto:
        errori.append("Il campo 'environment.schema' è obbligatorio.")
        return errori

    for tabella in config.get("tabelle", []):
        nome_tabella = tabella.get("name", "")
        colonne = tabella.get("columns", [])

        if nome_tabella != nome_tabella.lower():
            errori.append(f"Il nome della tabella '{nome_tabella}' deve essere in minuscolo.")
        if len(nome_tabella) > 35:
            errori.append(f"Il nome della tabella '{nome_tabella}' supera i 35 caratteri.")

        parts = nome_tabella.split("_", 2)
        if len(parts) < 3:
            errori.append(f"Il nome '{nome_tabella}' non segue la struttura '[codice]_[prefisso]_[descrizione]'.")
        else:
            codice, prefisso, descrizione = parts
            if codice != codice_prodotto:
                errori.append(f"Il codice prodotto '{codice}' in '{nome_tabella}' non corrisponde allo schema '{codice_prodotto}'.")
            if prefisso not in allowed_prefixes:
                errori.append(f"Il prefisso '{prefisso}' in '{nome_tabella}' non è valido (ammessi: {', '.join(allowed_prefixes)}).")
            if not descrizione:
                errori.append(f"La descrizione in '{nome_tabella}' è mancante.")

        nomi_colonne = {col.get("name", "") for col in colonne}
        missing = campi_tecnici - nomi_colonne
        if missing:
            errori.append(f"In '{nome_tabella}' mancano i campi tecnici: {', '.join(missing)}")

    return errori


def process_table(table_full_name: str, full_name: str, description_table: str, view_on_modellato_layer: str, schema_name: str) -> dict:
    """
    Legge lo schema Spark di una tabella Delta e costruisce la struttura dizionario
    pronta per essere serializzata in YAML.
    """
    df = spark.table(table_full_name)
    schema = df.schema

    desc_df = spark.sql(f"DESCRIBE TABLE EXTENDED {table_full_name}")
    desc_data = desc_df.collect()

    field_comments = {}
    field_masking = {}
    field_masking_comment = {}
    current_section = "columns"

    for row in desc_data:
        col_name = getattr(row, "col_name", None)
        data_type = getattr(row, "data_type", "")
        comment = getattr(row, "comment", "")
        if not col_name:
            continue
        if col_name.strip().startswith("#"):
            current_section = "masking" if "# Column Masks" in col_name else "other"
            continue
        if current_section == "columns":
            col_name_clean = col_name.strip()
            comment = comment.strip() if comment else ""
            if comment.upper().startswith("USING COLUMNS("):
                m = re.search(r"USING COLUMNS\('(.+?)'\)", comment)
                if m:
                    comment = m.group(1)
            field_comments[col_name_clean] = comment
            if col_name_clean not in field_masking:
                field_masking[col_name_clean] = None
        elif current_section == "masking":
            col_name_clean = col_name.strip()
            masking_func = (data_type.strip() if data_type else None)
            if masking_func:
                masking_func = masking_func.replace("`", "").replace("'", "")
            field_masking[col_name_clean] = masking_func
            if comment and comment.upper().startswith("USING COLUMNS("):
                m = re.search(r"USING COLUMNS\('(.+?)'\)", comment)
                if m:
                    field_masking_comment[col_name_clean] = m.group(1)

    columns = []
    possible_pk = None
    liquid_column = None

    for field in schema.fields:
        normalized_type = normalize_type(str(field.dataType))
        comment = field_comments.get(field.name, "")
        masking_value = field_masking.get(field.name, None)
        ad_group = field_masking_comment.get(field.name, None)
        columns.append({
            "name": field.name,
            "description": comment,
            "type": normalized_type,
            "nullable": field.nullable,
            "masked": masking_value if masking_value else None,
            "gdpr": True if masking_value else None,
            "active_dir_group": ad_group,
            "autoincrement": False
        })
        fname_lower = field.name.lower()
        if not possible_pk and (
            "id" in fname_lower or "pk" in fname_lower or
            (normalized_type in ["STRING", "INT", "BIGINT"] and not field.nullable)
        ):
            possible_pk = field.name
        if not liquid_column and normalized_type in ["DATE", "TIMESTAMP"] and any(
            x in fname_lower for x in ["data", "date", "dt", "time"]
        ):
            liquid_column = field.name

    if not any(col["name"].lower() == "iid_recordid" for col in columns):
        columns.insert(0, {
            "name": "iid_RecordID",
            "description": "Chiave autoincrementante",
            "type": "LONG",
            "nullable": False,
            "masked": False,
            "gdpr": False,
            "active_dir_group": None,
            "autoincrement": True
        })

    if not possible_pk and columns:
        possible_pk = columns[0]["name"]
    if not liquid_column:
        for field in schema.fields:
            if normalize_type(str(field.dataType)) in ["DATE", "TIMESTAMP"]:
                liquid_column = field.name
                break

    detail_df = spark.sql(f"DESCRIBE DETAIL {table_full_name}")
    detail = detail_df.collect()[0]
    delta_props = detail.asDict().get("properties", {})
    properties = {
        "delta.enableChangeDataFeed": delta_props.get("delta.enableChangeDataFeed", "false"),
        "delta.enableDeletionVectors": delta_props.get("delta.enableDeletionVectors", "false"),
        "delta.minReaderVersion": str(delta_props.get("delta.minReaderVersion", getattr(detail, 'minReaderVersion', ""))),
        "delta.minWriterVersion": str(delta_props.get("delta.minWriterVersion", getattr(detail, 'minWriterVersion', "")))
    }

    return {
        "name": full_name,
        "description": description_table,
        "primary_key": possible_pk,
        "is_liquid_cluster_prediction": True,
        "liquid_cluster": liquid_column if liquid_column else "",
        "properties": properties,
        "view_on_modellato_layer": view_on_modellato_layer,
        "columns": columns
    }


def table_exists(catalog_name: str, schema_name: str, table_name: str) -> bool:
    """Verifica se una tabella esiste nel catalogo/schema specificati."""
    full_table_name = f"{catalog_name}.{schema_name}.{table_name}"
    try:
        spark.sql(f"DESCRIBE TABLE EXTENDED {full_table_name}").collect()
        return True
    except Exception:
        print(f"La tabella '{full_table_name}' NON ESISTE")
        return False


In [0]:
config = read_yaml_file(config_path_yaml)
exclude_keys = ['user_list', 'service_principal_list', 'environment', 'logging', 'processor']

# Raccoglie gli oggetti (tabelle) definiti nel config_setup.yml
lista_oggetti_yml = []

for gruppo, val in config.items():
    print(f"\nGruppo {gruppo}")
    chiave_caricata = ""

    if gruppo not in exclude_keys:
        if isinstance(val, dict):
            for chiave, val2 in val.items():
                print(f"-- Chiave {chiave}")
                if isinstance(val2, list):
                    for oggetto in val2:
                        if oggetto.get('name') != chiave_caricata:
                            oggetto['Oggetto'] = chiave
                            chiave_caricata = oggetto['name']
                            lista_oggetti_yml.append(oggetto)
                elif isinstance(val2, dict):
                    val2["Oggetto"] = chiave
                    lista_oggetti_yml.append(val2)
        elif isinstance(val, list):
            for oggetto in val:
                if isinstance(oggetto, dict):
                    if oggetto.get('name') != chiave_caricata:
                        print(f"-- Chiave {oggetto.get('name')}")
                        oggetto['Oggetto'] = gruppo
                        chiave_caricata = oggetto['name']
                        lista_oggetti_yml.append(oggetto)
    else:
        print(f"Gruppo {gruppo} escluso")


In [0]:
# ============================================================
# MAIN: per ogni oggetto del config, se la tabella esiste
# in Sandbox, ne genera la mappatura YAML e la scrive su file.
# ============================================================

config = read_yaml_file(config_path_yaml)

# --- Lettura ambiente dal config ---
sbx_schema   = get_yaml_value(config, 'environment.dev.schema')
sbx_catalogo = get_yaml_value(config, 'environment.dev.catalog')
schema_dest  = get_yaml_value(config, 'environment.schema',  '')
catalogo_dest= get_yaml_value(config, 'environment.catalog', '')

# --- Diagnostica: mostra la sezione 'environment' così com'è nel config ---
env_raw = config.get('environment', {})
print("=" * 60)
print("Sezione 'environment' letta dal config:")
print(yaml.dump(env_raw, allow_unicode=True, default_flow_style=False))
print("=" * 60)

# --- Avviso se i valori sandbox non sono stati trovati ---
if not sbx_schema or not sbx_catalogo:
    print("⚠️  ATTENZIONE: 'environment.dev.schema' o 'environment.dev.catalog' non trovati nel config.")
    print("    Verifica la struttura del config sopra e aggiorna le chiavi di lettura se necessario.")
    print(f"    sbx_schema   = {sbx_schema}")
    print(f"    sbx_catalogo = {sbx_catalogo}")
    raise ValueError("Impossibile procedere: schema/catalogo sandbox non risolti. Correggi le chiavi YAML.")

print(f"Schema Sbx:    {sbx_schema}")
print(f"Catalogo Sbx:  {sbx_catalogo}")
print(f"Schema Dest:   {schema_dest}")
print(f"Catalogo Dest: {catalogo_dest}")

# Azzera il file di output
crea_file_vuoto_workspace(percorso_yml_generato)

errori_globali = []

for oggetto in lista_oggetti_yml:
    nome = oggetto['name']
    tab_exists = table_exists(sbx_catalogo, sbx_schema, nome)

    if not tab_exists:
        continue

    print(f"\nAnalisi tabella: {nome}")

    descr        = oggetto.get('description', '')
    view_setting = oggetto.get('view_on_modellato_layer', False)
    table_full   = f"{sbx_catalogo}.{sbx_schema}.{nome}"

    try:
        tab_config = process_table(
            table_full_name=table_full,
            full_name=nome,
            description_table=str(descr),
            view_on_modellato_layer=str(view_setting),
            schema_name=schema_dest
        )

        # Validazione
        yaml_dict_tmp = {"environment": {"schema": schema_dest}, "tabelle": [tab_config]}
        errori = validate_yaml_mapping(yaml_dict_tmp)
        print(errori)

        if errori:
            for err in errori:
                errori_globali.append(f"[{nome}] {err}")

        # Scrive sempre la tabella nel file (anche se ci sono warning di validazione)
        with open(percorso_yml_generato, "a", encoding="utf-8") as f:
            yaml.safe_dump([tab_config], f, sort_keys=False, allow_unicode=True,
                           default_flow_style=False, indent=2, width=4096)

        print(f" ")

    except Exception as e:
        print(f"❌ Errore durante l'elaborazione di '{nome}': {e}")
        errori_globali.append(f"[{nome}] Errore elaborazione: {e}")

print("\n" + "="*60)
print(f"✅ YAML scritto in: {percorso_yml_generato}")
if errori_globali:
    print(f"\n⚠️  Errori/Warning trovati ({len(errori_globali)}):")
    for e in errori_globali:
        print(f"  - {e}")
else:
    print("✅ Nessun errore di validazione.")


In [0]:
# ============================================================
# Legge la sezione 'modeling' dal config e scrive
# la sezione 'models:' nel file YAML di output.
#
# Gestisce due strutture possibili:
#
# Caso 1 - modello singolo con 'name' diretto:
#   modeling:
#     name: prchu_model
#     description: ...
#
# Caso 2 - sotto-chiavi ciascuna con il proprio 'name':
#   modeling:
#     model_deterministic:
#       name: creri_model_deterministic
#       description: ...
#     model_scoring:
#       name: creri_model_scoring
#       description: ...
# ============================================================

modeling = config.get('modeling', None)
elenco_models = ""

if modeling and isinstance(modeling, dict):
    if 'name' in modeling:
        # Caso 1: modello singolo con chiave 'name' diretta
        elenco_models += f"- name: {modeling['name']}\n"
    else:
        # Caso 2: sotto-chiavi, ognuna è un dizionario con 'name'
        for chiave, val in modeling.items():
            if isinstance(val, dict) and 'name' in val:
                elenco_models += f"- name: {val['name']}\n"

if elenco_models:
    with open(percorso_yml_generato, "a", encoding="utf-8") as f:
        f.write("\nmodels:\n")
        f.write(elenco_models)
    print(f"✅ Sezione 'models' aggiunta.")
else:
    print("ℹ️  Sezione 'modeling' non trovata o vuota nel config: nessun modello aggiunto.")


In [0]:
# ============================================================
# Aggiunge in coda al file YAML le sezioni
# 'online_tables' e 'feature_lookups' (se presenti)
# ============================================================

def pk_to_str(pk) -> str:
    """Normalizza il primary_key: se è una lista restituisce il primo elemento, altrimenti lo converte in stringa."""
    if isinstance(pk, list):
        return pk[0] if pk else ''
    return str(pk) if pk else ''


# --- Online Tables (oggetti con 'oltab' nel nome) ---
Elenco_Oltab = ""
for oggetto in lista_oggetti_yml:
    if "oltab" in oggetto['name'].lower():
        for obj in lista_oggetti_yml:
            if oggetto.get('src', '').lower() == obj['name'].lower():
                pk = pk_to_str(obj.get('primary_key', ''))
                Elenco_Oltab += f"- name: {oggetto['name']}\n"
                Elenco_Oltab += f"  source: {oggetto['src']}\n"
                Elenco_Oltab += f"  primary_key: {pk}\n"

if Elenco_Oltab:
    with open(percorso_yml_generato, "a", encoding="utf-8") as f:
        f.write("\nonline_tables:\n")
        f.write(Elenco_Oltab)
    print("✅ Sezione 'online_tables' aggiunta.")


# --- Feature Lookups (oggetti con 'udf_' nel nome) ---
Elenco_Feature = ""
for oggetto in lista_oggetti_yml:
    if "udf_" in oggetto['name'].lower():
        for obj in lista_oggetti_yml:
            if oggetto.get('src', '').lower() == obj['name'].lower():
                pk = pk_to_str(obj.get('primary_key', ''))
                Elenco_Feature += f"- name: {oggetto['name']}\n"
                Elenco_Feature += f"  table_name: {oggetto['src']}\n"
                Elenco_Feature += f"  lookup_key: {pk}\n"

if Elenco_Feature:
    with open(percorso_yml_generato, "a", encoding="utf-8") as f:
        f.write("\nfeature_lookups:\n")
        f.write(Elenco_Feature)
    print("✅ Sezione 'feature_lookups' aggiunta.")
